# 🔧 Notebook 02 — Data Preprocessing

**Purpose**: Clean data, handle scale differences, perform chronological splitting, address class imbalance, produce saved pipeline artifacts.

**Theory**: Lecture 02 Steps 2-3: Handle Missing Values and Transform Features. Lecture 01: Always split into Training, Validation, and Test sets.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
os.makedirs('../plots', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('Ready.')

In [ ]:
# Load data
df = pd.read_csv('../data/nev_battery_charging.csv')
print(f'Original shape: {df.shape}')
df.head()

## 1. Drop Timestamp

The timestamp column is a sequential integer 0-1899 with no predictive meaning. Its information is already encoded in battery_temp and SOC. Using it would teach the model to predict based on row number — **pure data leakage**.

In [ ]:
# Drop timestamp
print(f'timestamp sample: {df["timestamp"].head().tolist()}')
df = df.drop(columns=['timestamp'])
print(f'Shape after drop: {df.shape}')

## 2. Check & Remove Duplicates

In [ ]:
# Duplicates
dup_count = df.duplicated().sum()
print(f'Duplicate rows found: {dup_count}')
if dup_count > 0:
    df = df.drop_duplicates()
    print(f'Shape after dedup: {df.shape}')
else:
    print('No duplicates — clean dataset.')

## 3. Handle Missing Values

**Theory**: Lecture 02 Step 2 — Replace missing with mean, median, or mode. We use **median** because battery_temp and SOC are skewed — mean would be pulled by extreme values.

In [ ]:
# Missing values check
missing = df.isnull().sum()
print(f'Total missing: {missing.sum()}')
if missing.sum() > 0:
    print('Columns with missing values:')
    print(missing[missing > 0])
else:
    print('✅ Zero missing values found.')
    print('Building imputer anyway for inference pipeline robustness.')

## 4. Chronological Train/Val/Test Split

### ⚠️ CRITICAL: No Random Splitting!

The `over_temp_flag` transitions from 0→1 around row 900. Random splitting would put future states into training and past states into test — **data leakage**.

**Split ratios**: 70% Train (rows 0-1329) | 15% Val (1330-1614) | 15% Test (1615-1899)

In [ ]:
# Chronological split
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train = df.iloc[:train_end].copy()
df_val = df.iloc[train_end:val_end].copy()
df_test = df.iloc[val_end:].copy()

print(f'Train: {len(df_train)} rows (0 to {train_end-1})')
print(f'Val:   {len(df_val)} rows ({train_end} to {val_end-1})')
print(f'Test:  {len(df_test)} rows ({val_end} to {n-1})')

In [ ]:
# Visualize the split
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# over_temp_flag across splits
for split_df, label, color in [(df_train,'Train','#2196F3'),(df_val,'Val','#FF9800'),(df_test,'Test','#F44336')]:
    axes[0].scatter(split_df.index, split_df['over_temp_flag'], s=2, alpha=0.5, label=label, color=color)
axes[0].set_ylabel('over_temp_flag')
axes[0].set_title('Chronological Split — over_temp_flag Distribution', fontweight='bold')
axes[0].legend()

# cycle_degradation across splits
for split_df, label, color in [(df_train,'Train','#2196F3'),(df_val,'Val','#FF9800'),(df_test,'Test','#F44336')]:
    axes[1].scatter(split_df.index, split_df['cycle_degradation'], s=2, alpha=0.5, label=label, color=color)
axes[1].set_ylabel('cycle_degradation')
axes[1].set_xlabel('Row Index')
axes[1].set_title('Chronological Split — cycle_degradation Distribution', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../plots/preprocess_chronological_split.png', dpi=150, bbox_inches='tight')
plt.show()

### Feature-Target Separation

In [ ]:
# Separate features and targets
target_cols = ['cycle_degradation', 'over_temp_flag', 'over_voltage_flag']

X_train = df_train.drop(columns=target_cols)
X_val = df_val.drop(columns=target_cols)
X_test = df_test.drop(columns=target_cols)

y_reg_train = df_train['cycle_degradation']
y_reg_val = df_val['cycle_degradation']
y_reg_test = df_test['cycle_degradation']

y_cls_temp_train = df_train['over_temp_flag']
y_cls_temp_val = df_val['over_temp_flag']
y_cls_temp_test = df_test['over_temp_flag']

y_cls_volt_train = df_train['over_voltage_flag']
y_cls_volt_val = df_val['over_voltage_flag']
y_cls_volt_test = df_test['over_voltage_flag']

print(f'Feature columns ({len(X_train.columns)}): {list(X_train.columns)}')
print(f'\nTarget shapes: reg={y_reg_train.shape}, cls_temp={y_cls_temp_train.shape}, cls_volt={y_cls_volt_train.shape}')

## 5. Feature Scaling

**Theory**: Lecture 02 — Standardization converts to zero mean and unit variance. Scale-sensitive models (SVM, XGBoost) need this.

**Critical rule**: Fit scaler ONLY on X_train. Transform all splits using the same fitted scaler.

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # FIT only on train
X_val_scaled = scaler.transform(X_val)           # TRANSFORM with train params
X_test_scaled = scaler.transform(X_test)         # TRANSFORM with train params

print('Scaler fitted on training data only.')
print(f'Train scaled shape: {X_train_scaled.shape}')
print(f'\nScaled feature means (should be ~0):')
print(pd.Series(X_train_scaled.mean(axis=0), index=X_train.columns).round(6))

In [ ]:
# Visualize scaling effect
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Before scaling
axes[0].boxplot([X_train[c].values for c in X_train.columns[:8]], labels=X_train.columns[:8])
axes[0].set_title('Before Scaling (first 8 features)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# After scaling
axes[1].boxplot([X_train_scaled[:,i] for i in range(min(8,X_train_scaled.shape[1]))],
                labels=X_train.columns[:8])
axes[1].set_title('After StandardScaler (first 8 features)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Effect of StandardScaler on Feature Distributions', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../plots/preprocess_scaling_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Target Transformation (Log1p)

**Theory**: Lecture 02 Log Transform — reduces skewness in data with extreme ranges. cycle_degradation spans 0.0001 to 0.001 (an order of magnitude). Log transform compresses this into a more uniform distribution.

**Important**: Use `np.expm1()` to convert predictions back to original units for interpretable metrics.

In [ ]:
# Log1p transform for regression target
y_reg_train_log = np.log1p(y_reg_train)
y_reg_val_log = np.log1p(y_reg_val)
y_reg_test_log = np.log1p(y_reg_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y_reg_train, bins=40, color='#F44336', edgecolor='white', alpha=0.85)
axes[0].set_title('Original cycle_degradation (Train)', fontweight='bold')
axes[0].set_xlabel('cycle_degradation')

axes[1].hist(y_reg_train_log, bins=40, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[1].set_title('Log1p Transformed (Train)', fontweight='bold')
axes[1].set_xlabel('log1p(cycle_degradation)')

plt.suptitle('Target Transformation — Before vs After Log1p', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../plots/preprocess_target_transform.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Original skewness: {y_reg_train.skew():.3f}')
print(f'Log skewness: {y_reg_train_log.skew():.3f}')

## 7. Class Imbalance Handling

### over_temp_flag
Using `class_weight='balanced'` instead of SMOTE because SMOTE on time-series creates synthetic samples that violate temporal order — physically impossible battery states.

### over_voltage_flag
If < 20 positive examples: rule-based fallback (voltage > 4.15).

In [ ]:
# Class imbalance analysis — over_temp_flag
print('=== over_temp_flag ===')
print(f'Training distribution:')
print(y_cls_temp_train.value_counts())
print(f'\nClass ratio: {(y_cls_temp_train==0).sum() / max((y_cls_temp_train==1).sum(),1):.1f}:1')

weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_cls_temp_train)
class_weights = {0: weights[0], 1: weights[1]}
print(f'Balanced class weights: {class_weights}')

# Visualize class distribution across splits
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (y, title) in zip(axes, [(y_cls_temp_train,'Train'),(y_cls_temp_val,'Val'),(y_cls_temp_test,'Test')]):
    vc = y.value_counts()
    bars = ax.bar(vc.index.astype(str), vc.values, color=['#4CAF50','#F44336'])
    ax.set_title(f'{title} Set', fontweight='bold')
    for bar, v in zip(bars, vc.values):
        ax.text(bar.get_x()+bar.get_width()/2, v+5, str(v), ha='center', fontweight='bold')
plt.suptitle('over_temp_flag Distribution Across Splits', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../plots/preprocess_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# over_voltage_flag analysis
print('\n=== over_voltage_flag ===')
volt_pos_train = int(y_cls_volt_train.sum())
print(f'Training positives: {volt_pos_train} ({volt_pos_train/len(y_cls_volt_train)*100:.2f}%)')
if volt_pos_train < 20:
    print('\n⚠️ Fewer than 20 positive examples in training.')
    print('→ Cannot train a meaningful classifier.')
    print('→ Using rule-based fallback: flag = 1 if action_voltage > 4.15 OR terminal_voltage > 4.18')
    print('\nThis is professional engineering judgment: a physics-based threshold is more reliable')
    print('than a classifier trained on <20 positive examples.')

## 8. Save Artifacts

In [ ]:
# Save preprocessing artifacts
joblib.dump(scaler, '../models/preprocessor_pipeline.pkl')
print('Saved: models/preprocessor_pipeline.pkl')

# Save feature columns
feature_cols_list = list(X_train.columns)
with open('../models/feature_columns.json', 'w') as f:
    json.dump(feature_cols_list, f, indent=2)
print(f'Saved: models/feature_columns.json ({len(feature_cols_list)} features)')

# Save class weights
with open('../models/class_weights.json', 'w') as f:
    json.dump({str(k): float(v) for k, v in class_weights.items()}, f, indent=2)
print('Saved: models/class_weights.json')

# Save split indices for reproducibility
split_info = {'train_end': train_end, 'val_end': val_end, 'total': n}
with open('../models/split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)
print('Saved: models/split_info.json')

## 📋 Preprocessing Summary

| Step | Action | Rationale |
|------|--------|----------|
| Drop timestamp | Removed | Sequential index = data leakage |
| Duplicates | Checked & removed | Prevents distorted training |
| Missing values | Imputer built (0 missing found) | Pipeline robustness for inference |
| Split | Chronological 70/15/15 | Temporal data leakage prevention |
| Scaling | StandardScaler (fit on train only) | Different feature scales (0-97 vs 0-1 vs 30-1129) |
| Target transform | log1p on cycle_degradation | Reduce skewness for regression |
| Class imbalance | Balanced weights (not SMOTE) | SMOTE violates temporal order |
| over_voltage | Rule-based fallback | <20 positive examples |